### Cohort filtering (for LOS >= 1, hospital_expire_flag == 0, age >= 18)
Merging necessary columns from **admissions** and **patients**

In [1]:
import pandas as pd

ind_hosp = pd.read_csv('mimic-iv-3.1/hosp/admissions.csv.gz')
ind_hosp = ind_hosp.drop(columns=['deathtime', 'admit_provider_id', 'language', 'marital_status', 'edregtime', 'edouttime'])

ind_hosp['admittime'] = pd.to_datetime(ind_hosp['admittime'])
ind_hosp['dischtime'] = pd.to_datetime(ind_hosp['dischtime'])
ind_hosp['los'] = (ind_hosp['dischtime'] - ind_hosp['admittime']).dt.days

ind_hosp = ind_hosp[(ind_hosp['los'] >= 1) & (ind_hosp['hospital_expire_flag'] == 0)]
ind_hosp = ind_hosp.drop(columns=['hospital_expire_flag'])

In [2]:
patients = pd.read_csv('mimic-iv-3.1/hosp/patients.csv.gz').drop(columns=['dod', 'anchor_year_group'])

ind_hosp = ind_hosp.merge(patients, on='subject_id', how='left')
ind_hosp['age'] = ind_hosp['anchor_age'] + (ind_hosp['admittime'].dt.year - ind_hosp['anchor_year'])
ind_hosp = ind_hosp.drop(columns=['anchor_age', 'anchor_year'])

ind_hosp = ind_hosp[(ind_hosp['age'] >= 18)]
ind_hosp

,subject_id,hadm_id,admittime,dischtime,admission_type,admission_location,discharge_location,insurance,race,los,gender,age
0,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,EW EMER.,EMERGENCY ROOM,HOME,Medicaid,WHITE,1,F,52
1,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,EW EMER.,EMERGENCY ROOM,HOSPICE,Medicaid,WHITE,1,F,52
2,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,EW EMER.,EMERGENCY ROOM,HOME,Medicaid,WHITE,2,F,52
3,10000084,23052089,2160-11-21 01:56:00,2160-11-25 14:52:00,EW EMER.,WALK-IN/SELF REFERRAL,HOME HEALTH CARE,Medicare,WHITE,4,M,72
4,10000117,27988844,2183-09-18 18:10:00,2183-09-21 16:30:00,OBSERVATION ADMIT,WALK-IN/SELF REFERRAL,HOME HEALTH CARE,Medicaid,WHITE,2,F,57
...,...,...,...,...,...,...,...,...,...,...,...,...
415226,19999784,29956342,2121-01-31 00:00:00,2121-02-05 12:44:00,ELECTIVE,PHYSICIAN REFERRAL,HOME,Medicaid,BLACK/AFRICAN AMERICAN,5,M,59
415227,19999828,25744818,2149-01-08 16:44:00,2149-01-18 17:00:00,EW EMER.,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Medicaid,WHITE,10,F,48
415228,19999828,29734428,2147-07-18 16:23:00,2147-08-04 18:10:00,EW EMER.,PHYSICIAN REFERRAL,HOME HEALTH CARE,Medicaid,WHITE,17,F,46
415229,19999840,26071774,2164-07-25 00:27:00,2164-07-28 12:15:00,EW EMER.,EMERGENCY ROOM,HOME,Private,WHITE,3,M,58


In [3]:
ind_hosp.isna().sum()

subject_id                0
hadm_id                   0
admittime                 0
dischtime                 0
admission_type            0
admission_location        1
discharge_location    51044
insurance              4588
race                      0
los                       0
gender                    0
age                       0
dtype: int64

### Diagnoses mapping to ICD-10, calculation of Charlson, CCSR, Chronic Condition Indicator

In [4]:
from icdmappings import Mapper

diagnoses_icd = pd.read_csv('mimic-iv-3.1/hosp/diagnoses_icd.csv.gz')

mapper = Mapper()
diagnoses_icd['icd10_code'] = diagnoses_icd.apply(
    lambda row: mapper.map(row['icd_code'], source='icd9', target='icd10') 
                if row['icd_version'] == 9 
                else row['icd_code'],
    axis=1
)
diagnoses_icd = diagnoses_icd.drop(columns=['icd_code', 'icd_version'])
diagnoses_icd = diagnoses_icd.dropna(subset=['icd10_code'])

diagnoses_icd['ccsr_category'] = diagnoses_icd['icd10_code'].apply(
    lambda code: mapper.map(code, source='icd10', target='ccsr')
)
diagnoses_icd['is_chronic'] = diagnoses_icd['icd10_code'].apply(
    lambda code: mapper.map(code, source='icd10', target='ccir')
)
diagnoses_icd = diagnoses_icd.merge(ind_hosp[['age', 'hadm_id']], on='hadm_id', how='inner')

diagnoses_icd

,subject_id,hadm_id,seq_num,icd10_code,ccsr_category,is_chronic,age
0,10000032,22841357,1,B1921,INF007,True,52
1,10000032,22841357,2,R188,SYM006,False,52
2,10000032,22841357,3,D696,BLD006,False,52
3,10000032,22841357,4,E871,END011,False,52
4,10000032,22841357,5,J449,RSP008,True,52
...,...,...,...,...,...,...,...
5330941,19999987,23865745,7,I2510,CIR011,True,57
5330942,19999987,23865745,8,R569,NVS009,False,57
5330943,19999987,23865745,9,B961,INF003,False,57
5330944,19999987,23865745,10,H53469,EYE010,False,57


In [5]:
diag_summary = diagnoses_icd.groupby('hadm_id').agg(
    num_diagnoses=('hadm_id', 'size'),
    num_chronic=('is_chronic', 'sum')
).reset_index()

top15_ccsr = diagnoses_icd['ccsr_category'].value_counts().head(15).index.tolist()
print(f"Top-15 CCSR: {top15_ccsr}")

for cat in top15_ccsr:
    if pd.notna(cat):
        hadm_with_cat = diagnoses_icd[diagnoses_icd['ccsr_category'] == cat]['hadm_id'].unique()
        diag_summary[f'ccsr_{cat}'] = diag_summary['hadm_id'].isin(hadm_with_cat).astype(int)

Top-15 CCSR: ['FAC021', 'FAC025', 'CIR011', 'END010', 'CIR007', 'END011', 'END003', 'DIG004', 'CIR019', 'CIR017', 'CIR008', 'END009', 'BLD003', 'MBD002', 'GEN003']


In [6]:
from comorbidipy import comorbidity
import polars as pl

diagnoses_icd = pl.DataFrame(diagnoses_icd)
result = comorbidity(diagnoses_icd, id_col="hadm_id", code_col="icd10_code", age_col="age")
result

hadm_id,age,dementia,pud,metacanc,copd,rheumd,hp,rend,msld,cevd,chf,aids,canc,diab,diabwc,mld,ami,pvd,comorbidity_score
i64,i64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64
29611901,60,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0
25619619,88,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0.0
23326611,62,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,5.0
20170351,70,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,3.0
26535456,55,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
27212550,91,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0
22652958,68,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,4.0
26173120,63,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0


In [7]:
result = result.to_pandas()
diagnosis_features = diag_summary.merge(
    result[['hadm_id', 'comorbidity_score']], 
    on='hadm_id', 
    how='left'
)
print(diagnosis_features.columns)
diagnosis_features

Index(['hadm_id', 'num_diagnoses', 'num_chronic', 'ccsr_FAC021', 'ccsr_FAC025',
       'ccsr_CIR011', 'ccsr_END010', 'ccsr_CIR007', 'ccsr_END011',
       'ccsr_END003', 'ccsr_DIG004', 'ccsr_CIR019', 'ccsr_CIR017',
       'ccsr_CIR008', 'ccsr_END009', 'ccsr_BLD003', 'ccsr_MBD002',
       'ccsr_GEN003', 'comorbidity_score'],
      dtype='object')


,hadm_id,num_diagnoses,num_chronic,ccsr_FAC021,ccsr_FAC025,ccsr_CIR011,ccsr_END010,ccsr_CIR007,ccsr_END011,ccsr_END003,ccsr_DIG004,ccsr_CIR019,ccsr_CIR017,ccsr_CIR008,ccsr_END009,ccsr_BLD003,ccsr_MBD002,ccsr_GEN003,comorbidity_score
0,20000019,12,6,1,0,0,1,1,1,0,0,0,0,0,0,1,0,0,1.0
1,20000034,28,11,1,1,0,0,0,1,1,0,0,0,1,0,0,0,1,5.0
2,20000041,10,6,1,0,0,1,1,0,1,1,0,0,0,1,0,0,0,0.0
3,20000045,16,9,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,6.0
4,20000057,23,12,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
414918,29999745,7,3,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0.0
414919,29999803,30,14,1,0,0,1,0,1,0,0,1,1,1,0,1,1,0,4.0
414920,29999809,15,10,1,0,1,1,1,0,0,0,0,0,0,0,0,0,0,1.0
414921,29999828,8,4,0,0,0,1,1,1,0,0,0,0,0,1,0,0,0,0.0


In [8]:
ind_hosp = ind_hosp.merge(diagnosis_features, on="hadm_id", how='left')

### Procedures preprocessing (number of procedures)

In [9]:
procedures_icd = pd.read_csv('mimic-iv-3.1/hosp/procedures_icd.csv.gz')
procedures_icd = procedures_icd.groupby('hadm_id').size().reset_index(name='num_procedures')
ind_hosp = ind_hosp.merge(procedures_icd, on='hadm_id', how='left')
ind_hosp['num_procedures'] = ind_hosp['num_procedures'].fillna(0).astype(int)
ind_hosp

,subject_id,hadm_id,admittime,dischtime,admission_type,admission_location,discharge_location,insurance,race,los,...,ccsr_DIG004,ccsr_CIR019,ccsr_CIR017,ccsr_CIR008,ccsr_END009,ccsr_BLD003,ccsr_MBD002,ccsr_GEN003,comorbidity_score,num_procedures
0,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,EW EMER.,EMERGENCY ROOM,HOME,Medicaid,WHITE,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,1
1,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,EW EMER.,EMERGENCY ROOM,HOSPICE,Medicaid,WHITE,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,1
2,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,EW EMER.,EMERGENCY ROOM,HOME,Medicaid,WHITE,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0
3,10000084,23052089,2160-11-21 01:56:00,2160-11-25 14:52:00,EW EMER.,WALK-IN/SELF REFERRAL,HOME HEALTH CARE,Medicare,WHITE,4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0
4,10000117,27988844,2183-09-18 18:10:00,2183-09-21 16:30:00,OBSERVATION ADMIT,WALK-IN/SELF REFERRAL,HOME HEALTH CARE,Medicaid,WHITE,2,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
415226,19999784,29956342,2121-01-31 00:00:00,2121-02-05 12:44:00,ELECTIVE,PHYSICIAN REFERRAL,HOME,Medicaid,BLACK/AFRICAN AMERICAN,5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1
415227,19999828,25744818,2149-01-08 16:44:00,2149-01-18 17:00:00,EW EMER.,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Medicaid,WHITE,10,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,3
415228,19999828,29734428,2147-07-18 16:23:00,2147-08-04 18:10:00,EW EMER.,PHYSICIAN REFERRAL,HOME HEALTH CARE,Medicaid,WHITE,17,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,5
415229,19999840,26071774,2164-07-25 00:27:00,2164-07-28 12:15:00,EW EMER.,EMERGENCY ROOM,HOME,Private,WHITE,3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2


### Prescriptions preprocessing (number of MAIN + BASE drugs)

In [10]:
prescriptions = pd.read_csv('mimic-iv-3.1/hosp/prescriptions.csv.gz')
prescriptions

/var/folders/m0/9dr3jfm12bj1fkg2xjqh62jc0000gn/T/ipykernel_2843/3577613178.py:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  prescriptions = pd.read_csv('mimic-iv-3.1/hosp/prescriptions.csv.gz')


,subject_id,hadm_id,pharmacy_id,poe_id,poe_seq,order_provider_id,starttime,stoptime,drug_type,drug,...,gsn,ndc,prod_strength,form_rx,dose_val_rx,dose_unit_rx,form_val_disp,form_unit_disp,doses_per_24_hrs,route
0,10000032,22595853,12775705,10000032-55,55.0,P85UQ1,2180-05-08 08:00:00,2180-05-07 22:00:00,MAIN,Furosemide,...,008209,5.107901e+10,40mg Tablet,NaN,40,mg,1,TAB,1.0,PO/NG
1,10000032,22595853,18415984,10000032-42,42.0,P23SJA,2180-05-07 02:00:00,2180-05-07 22:00:00,MAIN,Ipratropium Bromide Neb,...,021700,4.879801e+08,2.5mL Vial,NaN,1,NEB,1,VIAL,4.0,IH
2,10000032,22595853,23637373,10000032-35,35.0,P23SJA,2180-05-07 01:00:00,2180-05-07 09:00:00,MAIN,Furosemide,...,008208,5.107901e+10,20mg Tablet,NaN,20,mg,1,TAB,1.0,PO/NG
3,10000032,22595853,26862314,10000032-41,41.0,P23SJA,2180-05-07 01:00:00,2180-05-07 01:00:00,MAIN,Potassium Chloride,...,001275,2.450041e+08,10mEq ER Tablet,NaN,40,mEq,4,TAB,1.0,PO
4,10000032,22595853,30740602,10000032-27,27.0,P23SJA,2180-05-07 00:00:00,2180-05-07 22:00:00,MAIN,Sodium Chloride 0.9% Flush,...,NaN,0.000000e+00,10 mL Syringe,NaN,3,mL,0.3,SYR,3.0,IV
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20292606,19999987,23865745,95605092,19999987-63,63.0,P213X2,2145-11-03 15:00:00,2145-11-03 18:00:00,MAIN,Propofol,...,65888.0,6.332303e+10,1000mg/100mL Vial,NaN,1000,mg,1,VIAL,0.0,IV DRIP
20292607,19999987,23865745,96309533,19999987-36,36.0,P59FJ8,2145-11-03 00:00:00,2145-11-04 13:00:00,BASE,0.9% Sodium Chloride,...,1210.0,3.380049e+08,100mL Bag,NaN,100,mL,100,mL,2.0,IV
20292608,19999987,23865745,96309533,19999987-36,36.0,P59FJ8,2145-11-03 00:00:00,2145-11-04 13:00:00,MAIN,LeVETiracetam,...,61005.0,1.478904e+10,500mg/5mL Vial,NaN,1000,mg,10,mL,2.0,IV
20292609,19999987,23865745,97298610,19999987-192,192.0,P47TKY,2145-11-08 16:00:00,2145-11-11 17:00:00,MAIN,Acetaminophen,...,4489.0,5.107900e+10,325mg Tablet,NaN,650,mg,2,TAB,NaN,PO/NG


In [11]:
prescriptions = prescriptions[prescriptions['drug_type'].isin(['MAIN', 'BASE'])]
meds_count = prescriptions.groupby('hadm_id')['drug'].nunique().reset_index(name='num_medications')
ind_hosp = ind_hosp.merge(meds_count, on='hadm_id', how='left')
ind_hosp['num_medications'] = ind_hosp['num_medications'].fillna(0).astype(int)

### Labevents preprocessing

In [12]:
import pandas as pd

necessary_columns = [
    'hadm_id',
    'itemid',
    'valuenum',
    'ref_range_lower',
    'ref_range_upper'
]

labevents = pd.read_csv('mimic-iv-3.1/hosp/labevents.csv.gz', usecols=necessary_columns)
labevents

,hadm_id,itemid,valuenum,ref_range_lower,ref_range_upper
0,NaN,50931,95.00,70.0,100.0
1,NaN,51071,NaN,NaN,NaN
2,NaN,51074,NaN,NaN,NaN
3,NaN,51075,NaN,NaN,NaN
4,NaN,51079,NaN,NaN,NaN
...,...,...,...,...,...
158374759,23865745.0,51279,3.52,4.2,5.4
158374760,23865745.0,51301,5.70,4.0,11.0
158374761,NaN,50912,1.10,0.4,1.1
158374762,NaN,50920,NaN,NaN,NaN


In [13]:
labevents = labevents.dropna(subset=['hadm_id'])

In [14]:
key_itemids = [
    # Chemistry
    50912,   # Creatinine
    51006,   # BUN
    50931,   # Glucose
    50983,   # Sodium
    50971,   # Potassium
    50902,   # Chloride
    50882,   # Bicarbonate
    50893,   # Calcium (Total)
    
    # Hematology
    50868,   # Hematocrit
    51222,   # Hemoglobin
    51301,   # WBC
    51265,   # Platelet Count
    51221,   # RDW
    51250,   # MCV
    51277    # MCH
]

labevents = labevents[labevents['itemid'].isin(key_itemids)]

In [15]:
labevents.isna().sum()

hadm_id                0
itemid                 0
valuenum           51451
ref_range_lower        0
ref_range_upper        0
dtype: int64

In [16]:
labevents = labevents.dropna(subset=['valuenum'])

In [17]:
lab_features = labevents.groupby(['hadm_id', 'itemid'])['valuenum'].agg(
    mean='mean',
    min='min',
    max='max', 
    last='last',
    count='count'
).reset_index()

labevents['is_abnormal'] = (
        (labevents['valuenum'] < labevents['ref_range_lower']) |
        (labevents['valuenum'] > labevents['ref_range_upper'])
    ).astype(int)
abnormal = labevents.groupby(['hadm_id', 'itemid'])['is_abnormal'].mean().reset_index()
abnormal.columns = ['hadm_id', 'itemid', 'abnormal_ratio']
lab_features = lab_features.merge(abnormal, on=['hadm_id', 'itemid'], how='left')

print(lab_features.head())

      hadm_id  itemid        mean    min    max   last  count  abnormal_ratio
0  20000019.0   50868   13.333333   12.0   14.0   14.0      3        0.000000
1  20000019.0   50882   26.000000   24.0   28.0   26.0      3        0.000000
2  20000019.0   50893    8.850000    8.8    8.9    8.9      2        0.000000
3  20000019.0   50902  101.666667  100.0  103.0  102.0      3        0.000000
4  20000019.0   50912    1.066667    0.9    1.2    0.9      3        0.333333


In [18]:
lab_pivot = lab_features.pivot_table(
    index='hadm_id',
    columns='itemid',
    values=['mean', 'min', 'max', 'last', 'count', 'abnormal_ratio']
)

lab_pivot.columns = [f'lab_{itemid}_{stat}' for stat, itemid in lab_pivot.columns]
lab_pivot = lab_pivot.reset_index()

lab_pivot

,hadm_id,lab_50868_abnormal_ratio,lab_50882_abnormal_ratio,lab_50893_abnormal_ratio,lab_50902_abnormal_ratio,lab_50912_abnormal_ratio,lab_50931_abnormal_ratio,lab_50971_abnormal_ratio,lab_50983_abnormal_ratio,lab_51006_abnormal_ratio,...,lab_50931_min,lab_50971_min,lab_50983_min,lab_51006_min,lab_51221_min,lab_51222_min,lab_51250_min,lab_51265_min,lab_51277_min,lab_51301_min
0,20000019.0,0.00,0.000000,0.000,0.000000,0.333333,1.000000,0.000000,0.000000,0.000000,...,169.0,3.5,136.0,14.0,23.9,8.3,86.0,183.0,12.3,5.8
1,20000024.0,0.00,0.000000,0.000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,...,88.0,5.2,140.0,28.0,32.1,10.3,98.0,196.0,14.8,4.9
2,20000034.0,0.00,0.000000,0.000,0.000000,1.000000,1.000000,0.000000,0.000000,1.000000,...,136.0,4.3,141.0,23.0,30.3,9.4,95.0,157.0,14.1,6.3
3,20000041.0,0.00,0.000000,NaN,0.000000,0.000000,1.000000,0.000000,0.000000,0.333333,...,116.0,3.9,136.0,18.0,24.5,8.3,88.0,229.0,13.1,7.4
4,20000045.0,0.00,0.250000,0.500,0.125000,0.125000,0.875000,0.250000,0.750000,0.250000,...,99.0,3.2,130.0,13.0,20.3,6.5,81.0,312.0,16.9,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
437837,29999803.0,0.25,0.285714,0.000,0.821429,0.000000,0.178571,0.142857,0.357143,0.964286,...,71.0,3.1,130.0,20.0,34.1,11.2,97.0,88.0,13.7,3.5
437838,29999809.0,0.00,0.375000,0.375,0.250000,0.125000,1.000000,0.250000,0.125000,0.000000,...,56.0,2.0,140.0,8.0,29.4,9.6,89.0,138.0,14.1,6.0
437839,29999828.0,0.00,0.000000,0.000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,...,213.0,4.3,135.0,10.0,37.5,12.2,89.0,182.0,13.6,8.7
437840,29999928.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,32.0,11.1,84.0,164.0,11.9,9.6


In [19]:
ind_hosp = ind_hosp.merge(lab_pivot, on='hadm_id', how='left')
lab_cols = [col for col in ind_hosp.columns if col.startswith('lab_')]
ind_hosp[lab_cols] = ind_hosp[lab_cols].fillna(0)

### Imputing missing values

In [20]:
cols_with_na = ind_hosp.columns[ind_hosp.isna().sum() > 0].tolist()
print("Columns with missing values:", cols_with_na)

na_counts = ind_hosp[cols_with_na].isna().sum()
print(na_counts[na_counts > 0])

Columns with missing values: ['admission_location', 'discharge_location', 'insurance', 'num_diagnoses', 'num_chronic', 'ccsr_FAC021', 'ccsr_FAC025', 'ccsr_CIR011', 'ccsr_END010', 'ccsr_CIR007', 'ccsr_END011', 'ccsr_END003', 'ccsr_DIG004', 'ccsr_CIR019', 'ccsr_CIR017', 'ccsr_CIR008', 'ccsr_END009', 'ccsr_BLD003', 'ccsr_MBD002', 'ccsr_GEN003', 'comorbidity_score']
admission_location        1
discharge_location    51044
insurance              4588
num_diagnoses           308
num_chronic             308
ccsr_FAC021             308
ccsr_FAC025             308
ccsr_CIR011             308
ccsr_END010             308
ccsr_CIR007             308
ccsr_END011             308
ccsr_END003             308
ccsr_DIG004             308
ccsr_CIR019             308
ccsr_CIR017             308
ccsr_CIR008             308
ccsr_END009             308
ccsr_BLD003             308
ccsr_MBD002             308
ccsr_GEN003             308
comorbidity_score       308
dtype: int64


In [21]:
ind_hosp['admission_location'] = ind_hosp['admission_location'].fillna('INFORMATION NOT AVAILABLE')
ind_hosp = ind_hosp[ind_hosp['discharge_location'] != 'DIED']
ind_hosp['discharge_location'] = ind_hosp['discharge_location'].fillna('UNKNOWN')
ind_hosp['insurance'] = ind_hosp['insurance'].fillna('UNKNOWN')
ind_hosp['insurance'] = ind_hosp['insurance'].replace({'No charge': 'Other'})

In [22]:
zero_fill_cols = [
    'num_diagnoses', 
    'num_chronic', 
    'comorbidity_score'
]
ccsr_cols = [col for col in ind_hosp.columns if col.startswith('ccsr_')]
zero_fill_cols.extend(ccsr_cols)

for col in zero_fill_cols:
    if col in ind_hosp.columns:
        ind_hosp[col] = ind_hosp[col].fillna(0).astype(int if col != 'comorbidity_score' else float)

### Target value (readmission in 30 days)

In [23]:
import pandas as pd

admissions_raw = pd.read_csv('mimic-iv-3.1/hosp/admissions.csv.gz')
admissions_raw['admittime'] = pd.to_datetime(admissions_raw['admittime'])
admissions_raw['dischtime'] = pd.to_datetime(admissions_raw['dischtime'])

admissions_raw = admissions_raw.sort_values(['subject_id', 'dischtime'])
admissions_raw['next_admittime'] = admissions_raw.groupby('subject_id')['admittime'].shift(-1)
admissions_raw['next_admission_type'] = admissions_raw.groupby('subject_id')['admission_type'].shift(-1)

admissions_raw['days_to_next'] = (
    admissions_raw['next_admittime'] - admissions_raw['dischtime']
).dt.days

planned_admission_types = ['ELECTIVE', 'SURGICAL SAME DAY ADMISSION']
admissions_raw['readmission_30day'] = (
    (admissions_raw['days_to_next'].notna()) &
    (admissions_raw['days_to_next'] <= 30) &
    (~admissions_raw['next_admission_type'].isin(planned_admission_types))
).astype(int)
admissions_raw['readmission_30day'].value_counts()

readmission_30day
0    442911
1    103117
Name: count, dtype: int64

In [24]:
ind_hosp = ind_hosp.merge(admissions_raw[['hadm_id', 'readmission_30day']],
                          on='hadm_id', how='left')

In [25]:
print(f"Readmission rate: {ind_hosp['readmission_30day'].mean():.2%}")
print(f"Distribution of target:\n{ind_hosp['readmission_30day'].value_counts()}")
print(f"Missing target: {ind_hosp['readmission_30day'].isna().sum()}")

Readmission rate: 19.45%
Distribution of target:
readmission_30day
0    334289
1     80739
Name: count, dtype: int64
Missing target: 0


### Time-based features (prior_admissions_12mo)

In [26]:
admissions = pd.read_csv('mimic-iv-3.1/hosp/admissions.csv.gz')
admissions['admittime'] = pd.to_datetime(admissions['admittime'])
admissions['dischtime'] = pd.to_datetime(admissions['dischtime'])


admissions_sorted = admissions.sort_values(['subject_id', 'admittime'])
admissions_sorted['prior_admissions_12mo'] = admissions_sorted.groupby('subject_id')['admittime'].transform(
    lambda x: [sum(1 for t in x.iloc[:i] if t >= x.iloc[i] - pd.Timedelta(days=365)) for i in range(len(x))]
)

In [27]:
ind_hosp = ind_hosp.merge(admissions_sorted[['hadm_id', 'prior_admissions_12mo']],
                          on='hadm_id', how='left')
ind_hosp

,subject_id,hadm_id,admittime,dischtime,admission_type,admission_location,discharge_location,insurance,race,los,...,lab_50983_min,lab_51006_min,lab_51221_min,lab_51222_min,lab_51250_min,lab_51265_min,lab_51277_min,lab_51301_min,readmission_30day,prior_admissions_12mo
0,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,EW EMER.,EMERGENCY ROOM,HOME,Medicaid,WHITE,1,...,126.0,29.0,35.5,12.4,99.0,137.0,15.7,6.6,1,1
1,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,EW EMER.,EMERGENCY ROOM,HOSPICE,Medicaid,WHITE,1,...,119.0,28.0,33.5,11.6,103.0,107.0,16.0,5.6,0,3
2,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,EW EMER.,EMERGENCY ROOM,HOME,Medicaid,WHITE,2,...,130.0,28.0,32.1,11.2,102.0,94.0,15.8,4.1,1,2
3,10000084,23052089,2160-11-21 01:56:00,2160-11-25 14:52:00,EW EMER.,WALK-IN/SELF REFERRAL,HOME HEALTH CARE,Medicare,WHITE,4,...,132.0,9.0,38.1,12.8,94.0,263.0,12.8,7.0,0,0
4,10000117,27988844,2183-09-18 18:10:00,2183-09-21 16:30:00,OBSERVATION ADMIT,WALK-IN/SELF REFERRAL,HOME HEALTH CARE,Medicaid,WHITE,2,...,143.0,9.0,39.3,13.2,94.0,180.0,12.6,8.8,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
415023,19999784,29956342,2121-01-31 00:00:00,2121-02-05 12:44:00,ELECTIVE,PHYSICIAN REFERRAL,HOME,Medicaid,BLACK/AFRICAN AMERICAN,5,...,137.0,4.0,35.2,11.5,83.0,220.0,13.1,2.7,0,4
415024,19999828,25744818,2149-01-08 16:44:00,2149-01-18 17:00:00,EW EMER.,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Medicaid,WHITE,10,...,133.0,7.0,28.3,8.4,73.0,324.0,17.3,8.6,0,0
415025,19999828,29734428,2147-07-18 16:23:00,2147-08-04 18:10:00,EW EMER.,PHYSICIAN REFERRAL,HOME HEALTH CARE,Medicaid,WHITE,17,...,136.0,5.0,28.7,8.8,79.0,252.0,15.4,6.4,0,0
415026,19999840,26071774,2164-07-25 00:27:00,2164-07-28 12:15:00,EW EMER.,EMERGENCY ROOM,HOME,Private,WHITE,3,...,138.0,7.0,38.6,14.6,84.0,198.0,14.2,12.8,0,0


### Categorical encoding

In [28]:
onehot_cols = [
    'admission_type',
    'discharge_location', 
    'insurance',
    'race',
    'admission_location'
]

binary_cols = ['gender']

ind_hosp = pd.get_dummies(
    ind_hosp,
    columns=onehot_cols,
    dummy_na=False,
    prefix=onehot_cols
)

ind_hosp['gender_male'] = (ind_hosp['gender'] == 'M').astype(int)
ind_hosp = ind_hosp.drop(columns=['gender', 'admittime'])
for col in onehot_cols:
    if col in ind_hosp.columns:
        ind_hosp = ind_hosp.drop(columns=[col])

In [29]:
ind_hosp.to_parquet('ind_hosp.parquet', index=False)